# Homework 6 — Genre SPLOM with linking and brushing

**Task.** Adapt the last chart in *Binding Selections to Multiple Inputs*,
preserving the rating option but creating a SPLOM to show the movie genres.
Include linking and brushing. Check all functionality works. Add a main title
and centre the rating selector underneath. Export a single HTML file.

The original binds one selection to two widgets — a genre dropdown and an MPAA
radio — and uses it to fade a single scatter plot. Here the genre dropdown is
replaced by *showing* genre: colour across a scatter plot matrix, with a
clickable legend. The rating radio is preserved.

In [1]:
"""Homework 6 - SPLOM of movie genres with a rating filter, linking and brushing."""
import re, altair as alt

movies = 'https://cdn.jsdelivr.net/npm/vega-datasets@1/data/movies.json'

MPAA = ['G', 'PG', 'PG-13', 'R', 'NC-17']
FIELDS = ['Rotten_Tomatoes_Rating', 'IMDB_Rating', 'Production_Budget']
LABELS = {'Rotten_Tomatoes_Rating': 'Rotten Tomatoes',
          'IMDB_Rating': 'IMDB', 'Production_Budget': 'Budget ($)'}

## Three selections, three jobs

- **`rating`** — the preserved widget, now a radio over MPAA ratings alone. It
  drives a `transform_filter`, so it removes films from the data rather than
  fading them.
- **`brush`** — one interval selection with `resolve='global'`, registered on
  *every* panel. Because all panels share the one parameter, dragging in any
  cell highlights the same films everywhere else. That is the linking and
  brushing.
- **`genre`** — a point selection bound to the legend, so clicking a genre
  isolates it across the whole matrix.

The last two are combined with `brush & genre` in the conditional encodings, so
a film must satisfy both to stay highlighted.

In [2]:
# --- selections --------------------------------------------------------------
# the rating widget the assignment asks to preserve, now a radio over MPAA only
rating = alt.selection_point(
    name='Rating', fields=['MPAA_Rating'], value=[{'MPAA_Rating': 'PG-13'}],
    bind=alt.binding_radio(options=MPAA, name='MPAA rating   '))

# one interval shared by every panel: brush in any cell, highlight everywhere
brush = alt.selection_interval(name='brush', resolve='global')

# clicking a genre in the legend isolates it across the whole matrix
genre = alt.selection_point(name='genre', fields=['Major_Genre'], bind='legend')

genre_scale = alt.Scale(scheme='tableau20')

## The matrix

`repeat()` is the usual way to build a SPLOM, but it always produces a full
square grid, cannot label axes per field, and would show every pair twice either
side of a redundant diagonal. Building the lower triangle explicitly gives each
pair once, with the x field constant down each column and the y field constant
across each row, and axis labels only on the outer edges.

The dataset also contains rows with no genre, which otherwise appear as a
`null` swatch in the legend — hence the `isValid` filter.

In [3]:
# --- the scatter plot matrix -------------------------------------------------
# repeat() always produces a full square grid and cannot label axes per field,
# so the lower triangle is built explicitly: every pair appears once, the x
# field is constant down each column and the y field constant across each row.
def cell(xf, yf, first_col, last_row):
    return alt.Chart(movies).mark_circle(size=22).encode(
        alt.X(xf, type='quantitative', scale=alt.Scale(zero=False),
              axis=alt.Axis(title=LABELS[xf] if last_row else None,
                            labels=last_row)),
        alt.Y(yf, type='quantitative', scale=alt.Scale(zero=False),
              axis=alt.Axis(title=LABELS[yf] if first_col else None,
                            labels=first_col)),
        color=alt.condition(brush & genre,
                            alt.Color('Major_Genre:N', scale=genre_scale,
                                      legend=alt.Legend(title='Major genre',
                                                        columns=2, symbolLimit=0)),
                            alt.value('#d9d9d9')),
        opacity=alt.condition(brush & genre, alt.value(0.75), alt.value(0.12)),
        size=alt.condition(brush & genre, alt.value(30), alt.value(14)),
        tooltip=['Title:N', 'Major_Genre:N', 'MPAA_Rating:N',
                 alt.Tooltip('Rotten_Tomatoes_Rating:Q', title='Rotten Tomatoes'),
                 alt.Tooltip('IMDB_Rating:Q', title='IMDB'),
                 alt.Tooltip('Production_Budget:Q', title='Budget', format='$,.0f')],
    ).add_params(brush).transform_filter(
        rating
    ).transform_filter(
        # the dataset has rows with no genre; they would show as a "null"
        # swatch in the legend
        'isValid(datum.Major_Genre)'
    ).properties(width=200, height=200)

rows = []
for r, yf in enumerate(FIELDS[1:], start=1):
    cols = [cell(xf, yf, first_col=(c == 0), last_row=(r == len(FIELDS) - 1))
            for c, xf in enumerate(FIELDS[:r])]
    rows.append(alt.hconcat(*cols, spacing=16))
splom = alt.vconcat(*rows, spacing=16)

chart = splom.add_params(rating, genre).properties(
    title=alt.TitleParams(
        text='Movie genres by critics, audience and budget',
        subtitle=['Drag inside any panel to brush — the selection highlights the '
                  'same films in every other panel. Click a genre in the legend '
                  'to isolate it.',
                  'The radio buttons below filter by MPAA rating.'],
        anchor='middle', fontSize=18, subtitleColor='gray', subtitleFontSize=12,
        offset=12),
).resolve_scale(color='shared').configure_view(stroke=None)

In [4]:
chart

alt.VConcatChart(...)

## Centring the rating selector

Vega-Embed renders bound inputs into a `.vega-bindings` block below the chart,
left-aligned, and there is no Altair option to change that — it is page layout,
not chart spec. The assignment asks for the selector centred underneath, so the
saved HTML gets a small stylesheet that flexes the bindings block to centre.

In [5]:
# --- export ------------------------------------------------------------------
# Vega-Embed renders bound inputs in a .vega-bindings block, left-aligned by
# default. The assignment asks for the rating selector centred underneath, so
# the saved page gets a small stylesheet.
CENTRE_CSS = """
<style>
  body { font-family: system-ui, -apple-system, "Segoe UI", sans-serif;
         margin: 0; padding: 24px; }
  #vis { display: flex; flex-direction: column; align-items: center; }
  .vega-bindings { display: flex; justify-content: center; align-items: center;
                   flex-wrap: wrap; gap: .35em; margin-top: 18px;
                   font-size: 14px; }
  .vega-bind { display: flex; align-items: center; gap: .3em; }
  .vega-bind label { display: inline-flex; align-items: center; gap: .25em;
                     margin-right: .6em; }
</style>
"""

def save_centred(ch, path):
    ch.save(path)
    html = open(path, encoding='utf-8').read()
    html = html.replace('</head>', CENTRE_CSS + '</head>') if '</head>' in html \
        else CENTRE_CSS + html
    open(path, 'w', encoding='utf-8').write(html)

In [6]:
save_centred(chart, 'solution.html')
print('saved solution.html')

saved solution.html


## Checking it works

Verified in the browser rather than assumed:

- dragging inside any panel draws a brush and fades unselected films in **all**
  panels — confirmed by brushing Rotten Tomatoes 40–70 / IMDB 6–7.5 and watching
  the highlighted points in both lower panels collapse to exactly that range
- the MPAA radio filters the data — selecting NC-17 leaves a handful of films
  and the legend shrinks to only the genres still present
- the radio buttons render centred beneath the chart
- clicking a legend entry isolates that genre